In [1]:
import frust as ft
from rdkit import Chem
from rdkit.Geometry import Point3D
from rdkit.Chem import AllChem, RWMol
# from frust.tsguess.specs import BUILTIN_TS_SPECS
from d60_tsguess2 import build_ts1_template_smiles, build_ts3_template_smiles

In [2]:
ts1_smiles = build_ts1_template_smiles(
    catalyst_smiles="BC1=C(N(C)C)C=CC=C1",
    substrate_smiles="CN1C=CC=C1",
)

ft.MolTo3DGrid(list(ts1_smiles.values()))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [3]:
ts1_smiles

{2: '[H][B-]([H])(c1ccccc1[N+]([H])(C)C)c1cccn1C',
 3: '[H][B-]([H])(c1ccn(C)c1)c1ccccc1[N+]([H])(C)C'}

In [4]:
ft.MolTo3DGrid("[H][B-]([H])(c1ccccc1[N+]([H])(C)C)c1cccn1C")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# TS1

In [5]:
mol = Chem.MolFromSmiles(ts1_smiles[3])
mol = Chem.AddHs(mol)

query = Chem.MolFromSmarts('[#1]~[#7]~[*]~[*]~[#5]~[#6]')

matches = mol.GetSubstructMatches(query)
print(matches)

H = matches[0][0]
N = matches[0][1]
B = matches[0][4]
C = matches[0][5]

coord_map = {
    H: Point3D(-1.558624, 0.1036, 1.047895),
    B: Point3D(0.373318, -0.291503, 1.700016),
    N: Point3D(-2.416144, -1.101445, 0.730403),
    C: Point3D(-0.659429, 0.98644, 1.328243)
}

AllChem.EmbedMultipleConfs(mol,100,
        maxAttempts=0,
        randomSeed=0xF00D,
        useRandomCoords=False,
        pruneRmsThresh=0.3,
        coordMap=coord_map,
        ignoreSmoothingFailures=True,
        enforceChirality=True,
        useSmallRingTorsions=True)

((28, 13, 12, 7, 0, 1),)


In [6]:
ft.MolTo3DGrid(mol, show_charges=False)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# TS2

In [7]:
mol = Chem.MolFromSmiles(ts1_smiles[2])
mol = Chem.AddHs(mol)

query = Chem.MolFromSmarts("[#1]~[#7]~[*]~[*]~[#5]~[#6]")

matches = mol.GetSubstructMatches(query)
print(matches)

NH = matches[0][0]
N = matches[0][1]
B = matches[0][4]
C = matches[0][5]

B_Hs = [
    atom.GetIdx()
    for atom in mol.GetAtomWithIdx(B).GetNeighbors()
    if atom.GetAtomicNum() == 1
]

BH = B_Hs[0]


coord_map = {
    NH: Point3D(5.52964600, 1.87819600, -0.90718600),
    B: Point3D(4.50562100, 2.68757200, 0.44199300),
    N: Point3D(5.17325900, 0.04177700, -1.00370400),
    BH: Point3D(5.64204400, 2.65108000, -0.82017600),
}

conf_ids = AllChem.EmbedMultipleConfs(mol,100,
        maxAttempts=0,
        randomSeed=0xF00D,
        useRandomCoords=False,
        pruneRmsThresh=0.5,
        coordMap=coord_map,
        ignoreSmoothingFailures=True,
        enforceChirality=True,
        useSmallRingTorsions=True)

len(conf_ids)

((22, 7, 6, 1, 0, 10),)


9

In [8]:
ft.MolTo3DGrid(mol, cell_size=(300,300), show_charges=False)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# TS3

In [9]:
ts3_smiles = build_ts3_template_smiles(
    catalyst_smiles="BC1=C(N(C)C)C=CC=C1",
    substrate_smiles="CN1C=CC=C1",
)
print(ts3_smiles)
ft.MolTo3DGrid(list(ts3_smiles.values()), show_charges=True)

{2: '[H][B-]1(c2ccccc2N(C)C)[H+][B-]2(OC(C)(C)C(C)(C)O2)C12[CH+]C=CN2C', 3: '[H][B-]1(c2ccccc2N(C)C)[H+][B-]2(OC(C)(C)C(C)(C)O2)C12C=CN(C)[CH+]2'}


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
ft.MolTo3DGrid("CN(C)C1=C([B-]2([H])C3([B-]4([H+]2)OC(C)(C)C(C)(C)O4)CC=C[NH+]3C)C=CC=C1")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [48]:
# CN(C1=C(C=CC=C1)[B-]2(C3([CH+]C=CN3C)[B-]4(OC(C)(C(C)(O4)C)C)[H+]2)[H])C
mol = Chem.MolFromSmiles(ts3_smiles[2])
mol = Chem.AddHs(mol)

# dimethyl_N - aryl_C - aryl_C - B
query = Chem.MolFromSmarts("[#5]~[#1]~[#5]~[#6]")

matches = mol.GetSubstructMatches(query)
print(matches)

B_cat = matches[0][0]
H_bridge = matches[0][1]
B_hbpin = matches[0][2]
C_ring = matches[0][3]

B_Hs = [
    atom.GetIdx()
    for atom in mol.GetAtomWithIdx(B_cat).GetNeighbors()
    if atom.GetAtomicNum() == 1
]

H_cat = next(idx for idx in B_Hs if idx != H_bridge)

print(H_cat)

coord_map = {
    B_cat: Point3D(1.20156300, 0.08036600, 0.66019900),
    H_cat: Point3D(1.583281, 0.889608, -0.137320),
    H_bridge: Point3D(1.67696200, -1.00450700, -0.05268600),
    B_hbpin: Point3D(2.53230800, -1.42833600, 0.77757800),
    C_ring: Point3D(1.97667200, 0.24890600, 2.06849400),
}

conf_ids = AllChem.EmbedMultipleConfs(mol,10,
        maxAttempts=0,
        randomSeed=0xF00D,
        useRandomCoords=False,
        pruneRmsThresh=-1,
        coordMap=coord_map,
        ignoreSmoothingFailures=True,
        enforceChirality=True,
        useSmallRingTorsions=True)

len(conf_ids)

((0, 10, 11, 20), (11, 10, 0, 1))
26


10

In [44]:
ft.MolTo3DGrid(mol, show_charges=False)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# TS4

In [13]:
mol = Chem.MolFromSmiles(ts3_smiles[3])
mol = Chem.AddHs(mol)

# dimethyl_N - aryl_C - aryl_C - B
query = Chem.MolFromSmarts("[#5]~[#1]~[#5]~[#6]")

matches = mol.GetSubstructMatches(query)
print(matches)

B_cat = matches[0][0]
H_bridge = matches[0][1]
B_hbpin = matches[0][2]
C_ring = matches[0][3]

coord_map = {
    B_cat: Point3D(-0.93003800, 0.59038400, 1.92979300),
    H_bridge: Point3D(-0.08788400, 1.26200500, 1.34482600),
    B_hbpin: Point3D(0.99948300, 1.21736900, 2.68353800),
    C_ring: Point3D(0.01306500, 0.44616100, 3.67687400),
}

conf_ids = AllChem.EmbedMultipleConfs(mol,10,
        maxAttempts=0,
        randomSeed=0xF00D,
        useRandomCoords=False,
        pruneRmsThresh=0.5,
        coordMap=coord_map,
        ignoreSmoothingFailures=True,
        enforceChirality=True,
        useSmallRingTorsions=True)

len(conf_ids)

((0, 10, 11, 20), (11, 10, 0, 1))


9

In [14]:
ft.MolTo3DGrid(mol, show_charges=True)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.